# 9.2 Agenda Analysis — Policy Topics Across Platforms

Uses the keyword dictionary from `9.1_agenda_scraping.ipynb` to track how Trump's and Harris's
policy agendas were covered across Reddit, Bluesky, and newspapers.

**Three analyses:**
1. **Topic salience over time** — which agenda topics spiked when?
2. **Platform × topic heatmap** — which platform/leaning amplified which candidate's agenda?
3. **Framing divergence** — same topic, very different vocabulary (TrumpBuzz vs HarrisBuzz)

**Input:** `Data/agenda_keywords.json` · Silver parquets / CSVs


<!-- toc -->
## Contents
- **[1. Setup & Load Agenda](#1-setup--load-agenda)**
- **[2. Load & Prepare Data](#2-load--prepare-data)**
- **[3. Topic Salience Over Time](#3-topic-salience-over-time)**
- **[4. Platform × Topic Heatmap](#4-platform--topic-heatmap)**
- **[5. Framing Divergence](#5-framing-divergence)**
- **[6. Summary](#6-summary)**


## 1. Setup & Load Agenda


In [1]:
import sys, json, ast
sys.path.insert(0, '../..')
from house_style import (
    apply_style, styled_fig, style_ax,
    BG_DARK, BG_PANEL, REPUBLICAN, DEMOCRAT, NEUTRAL,
    TEXT_PRIMARY, TEXT_MUTED, GRID_COLOR, SPINE_COLOR, PALETTE,
    BUZZ_COLORS, EVENTS, add_events, event_legend_handles,
    C_VIX,
    C_SP500,
    C_FEAR,
    C_ANGER,
    C_TRUST,
    C_DISGUST,
    C_SADNESS,
    C_JOY,
    C_ANTICIPATION,
)
apply_style()

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.dates as mdates
from sklearn.feature_extraction.text import TfidfVectorizer

DATA_DIR  = Path('../../Data/2_Silver')
AGENDA_PATH = Path('../../Data/agenda_keywords.json')

In [2]:
# Load agenda keywords (run 9.1 first)
with open(AGENDA_PATH, encoding='utf-8') as f:
    agenda = json.load(f)

# Flatten to a single topic dict with candidate ownership
# Format: {topic_key: {candidate, title, keywords}}
topics = {}
for cand in ('harris', 'trump'):
    for key, data in agenda[cand].items():
        full_key = f"{cand}_{key}"
        topics[full_key] = {
            'candidate': cand,
            'title':     data['title'],
            'keywords':  [k.lower() for k in data['keywords'][:12]],
        }

# Also build a compact cross-candidate view of shared themes
SHARED_THEMES = {
    'Economy':       {'trump': ['trump_economy_trade', 'trump_taxes'],
                      'harris': ['harris_economy']},
    'Immigration':   {'trump': ['trump_immigration'], 'harris': ['harris_safety_justice']},
    'Energy':        {'trump': ['trump_energy_environment'], 'harris': ['harris_foreign_security']},
    'Reproductive\nrights': {'trump': ['trump_abortion'], 'harris': ['harris_freedoms']},
    'Crime / Justice': {'trump': ['trump_crime'], 'harris': ['harris_safety_justice']},
    'Foreign policy':  {'trump': ['trump_foreign_defense'], 'harris': ['harris_foreign_security']},
}

print(f'Agenda loaded: {len(agenda["harris"])} Harris sections, {len(agenda["trump"])} Trump sections')
print()
for key, meta in list(topics.items())[:8]:
    print(f'  {key:35s}: {", ".join(meta["keywords"][:6])}')
print('  ...')


Agenda loaded: 4 Harris sections, 50 Trump sections

  harris_economy                     : class, middle class, tax, small, million, jobs
  harris_freedoms                    : abortion, reproductive, voting, love, women, lgbtqi
  harris_safety_justice              : border, gun, violence, fentanyl, gun violence, immigration
  harris_foreign_security            : stand, leaders, national security, veterans, russia, israel
  trump_campaign_finance             : 107, pacs, super pacs, super, race, early 2016
  trump_civil_servants               : deconstruct administrative, bannon stated, briefly, briefly leader, chief strategist, christie
  trump_disabled_people              : disability, 119, disability related, disability group, disabled people, website mention
  trump_education                    : school, charter, education, school choice, schoolers, choice
  ...


## 2. Load & Prepare Data


In [ ]:
# ── Reddit posts ──────────────────────────────────────────────────────────────
reddit = pd.read_parquet(DATA_DIR / 'Reddit/reddit_posts_clean.parquet')
reddit['date'] = pd.to_datetime(reddit['created_utc'], utc=True).dt.tz_localize(None).dt.normalize()
reddit['week'] = reddit['date'].dt.to_period('W').dt.start_time
reddit['text_clean'] = reddit['text_clean'].fillna('').astype(str)
reddit['platform'] = 'Reddit'
reddit['group'] = reddit['candidate']           # TrumpBuzz / HarrisBuzz / ElectionBuzz

# ── Bluesky posts ─────────────────────────────────────────────────────────────
bsky = pd.read_csv(DATA_DIR / 'Bluesky/bluesky_clean.csv', low_memory=False)
bsky['date'] = pd.to_datetime(bsky['timestamp'], utc=True).dt.tz_localize(None).dt.normalize()
bsky['week'] = bsky['date'].dt.to_period('W').dt.start_time
bsky['text_clean'] = bsky['text_clean'].fillna('').astype(str)
bsky['platform'] = 'Bluesky'
bsky['group'] = bsky['candidate']

# ── Newspapers ────────────────────────────────────────────────────────────────
news = pd.read_csv(DATA_DIR / 'Newspapers/mediacloud_articles_clean.csv', low_memory=False)
news['date'] = pd.to_datetime(news['date'])
news['week'] = news['date'].dt.to_period('W').dt.start_time
news['text_clean'] = news['title_clean'].fillna('').astype(str)
news['platform'] = 'News'
news['group'] = news['leaning'].map({
    'Democratic':     'HarrisBuzz',
    'Republican':     'TrumpBuzz',
    'Center/Unknown': 'ElectionBuzz',
}).fillna('ElectionBuzz')

# ── Combined ──────────────────────────────────────────────────────────────────
COLS = ['date', 'week', 'platform', 'group', 'text_clean']
combined = pd.concat([
    reddit[COLS], bsky[COLS], news[COLS]
], ignore_index=True)

combined = combined[
    (combined['date'] >= '2024-07-05') &
    (combined['date'] <= '2024-11-04')
].reset_index(drop=True)

print(f'Combined dataset: {len(combined):,} rows')
print(combined.groupby(['platform', 'group']).size().to_string())


ValueError: time data "2024-11-04 21:44:50+00:00" doesn't match format "%Y-%m-%d %H:%M:%S.%f%z", at position 7. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

: 

In [ ]:
# Tag each row with which agenda topics it mentions
# For speed: build one regex pattern per topic
import re

def build_pattern(keywords):
    """OR-regex from keyword list, matching whole words."""
    escaped = [re.escape(k) for k in keywords if k]
    return re.compile(r'\b(' + '|'.join(escaped) + r')\b', re.IGNORECASE) if escaped else None

topic_patterns = {
    key: build_pattern(meta['keywords'])
    for key, meta in topics.items()
}

# Apply: binary hit columns
texts = combined['text_clean'].values
for key, pat in topic_patterns.items():
    if pat:
        combined[key] = [bool(pat.search(t)) for t in texts]
    else:
        combined[key] = False

topic_cols = list(topic_patterns.keys())
hits = combined[topic_cols].sum()
print('Total keyword hits per topic (top 10):')
print(hits.sort_values(ascending=False).head(10).to_string())


## 3. Topic Salience Over Time


For each week, what fraction of posts mentions each agenda topic?  
We show the **top-5 most-mentioned topics per candidate** to keep the chart readable.  
Dashed vertical lines mark the key political events.


In [ ]:
# Select top-N topics by total hit count
TOP_N = 5

harris_topics = [k for k in topic_cols if k.startswith('harris_')]
trump_topics  = [k for k in topic_cols if k.startswith('trump_')]

top_harris = hits[harris_topics].nlargest(TOP_N).index.tolist()
top_trump  = hits[trump_topics ].nlargest(TOP_N).index.tolist()

# Weekly hit rate per topic (fraction of posts that week)
weekly_total = combined.groupby('week').size().rename('total')

def weekly_rate(topic_key):
    hits_w = combined.groupby('week')[topic_key].sum()
    return (hits_w / weekly_total * 100).fillna(0)

weeks = sorted(combined['week'].unique())

print('Top Harris topics:', top_harris)
print('Top Trump topics: ', top_trump)


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 10), facecolor=BG_DARK, sharex=True)
fig.suptitle('Agenda Topic Salience Over Time\n(% of all posts mentioning topic per week)',
             color=TEXT_PRIMARY, fontsize=14, fontweight='bold', y=0.98)

harris_palette = [DEMOCRAT, '#5dade2', '#2e86c1', '#85c1e9', '#1a5276']
trump_palette  = [REPUBLICAN, '#e07b39', '#c0392b', '#f1948a', '#922b21']

for ax, topic_list, palette, cand_label in [
    (axes[0], top_harris, harris_palette, 'Harris agenda'),
    (axes[1], top_trump,  trump_palette,  'Trump agenda'),
]:
    ax.set_facecolor(BG_PANEL)
    handles = []
    for i, key in enumerate(topic_list):
        rate = weekly_rate(key)
        label = topics[key]['title'][:30]
        color = palette[i % len(palette)]
        line, = ax.plot(rate.index, rate.values, color=color,
                        linewidth=2.2, label=label)
        ax.fill_between(rate.index, rate.values, alpha=0.08, color=color)
        handles.append(line)

    add_events(ax)
    ax.set_ylabel('% of posts', color=TEXT_MUTED, fontsize=10)
    ax.set_title(cand_label, color=TEXT_PRIMARY, fontsize=12, loc='left', pad=8)
    ax.tick_params(colors=TEXT_MUTED)
    ax.grid(axis='y', color=GRID_COLOR, linewidth=0.7)
    ax.set_axisbelow(True)
    for sp in ['top', 'right']:
        ax.spines[sp].set_visible(False)
    ax.spines['left'].set_edgecolor(SPINE_COLOR)
    ax.spines['bottom'].set_edgecolor(SPINE_COLOR)
    ax.legend(handles=handles, loc='upper left', fontsize=8,
              facecolor=BG_PANEL, edgecolor=SPINE_COLOR, labelcolor=TEXT_PRIMARY)

axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
axes[1].xaxis.set_major_locator(mdates.MonthLocator())
axes[1].tick_params(axis='x', colors=TEXT_MUTED)

# Shared event legend at bottom
fig.legend(
    handles=event_legend_handles(),
    loc='lower center', bbox_to_anchor=(0.5, 0.0),
    ncol=5, facecolor=BG_PANEL, edgecolor=SPINE_COLOR,
    labelcolor=TEXT_PRIMARY, fontsize=8, framealpha=0.95
)
plt.tight_layout(rect=[0, 0.07, 1, 0.97])
plt.savefig('../../Data/2_Silver/agenda_salience_over_time.png',
            dpi=150, bbox_inches='tight', facecolor=BG_DARK)
plt.show()


**Insight — Topic salience over time:**  
Identify which events caused agenda-topic spikes. For instance: did Biden's withdrawal spike Harris's
'freedoms/abortion' topic? Did the debate spike 'economy' for Trump? Do the spikes align with Polymarket odds shifts?


## 4. Platform × Topic Heatmap


Which platform/audience segment amplifies which candidate's agenda most?  
Rows = platform × buzz group. Columns = agenda topics (top per candidate).  
Values = % of posts in that segment mentioning the topic — reveals **selective amplification**.


In [ ]:
# Segments: platform × group (only partisan groups)
SEGMENTS = [
    ('Reddit',  'TrumpBuzz'),
    ('Reddit',  'HarrisBuzz'),
    ('Bluesky', 'TrumpBuzz'),
    ('Bluesky', 'HarrisBuzz'),
    ('News',    'TrumpBuzz'),
    ('News',    'HarrisBuzz'),
]

# Columns: top-4 Harris + top-4 Trump topics
TOP_HM = 4
hm_cols = top_harris[:TOP_HM] + top_trump[:TOP_HM]
col_labels = [topics[k]['title'][:22] for k in hm_cols]
seg_labels = [f'{p}\n{g.replace("Buzz","")}' for p, g in SEGMENTS]

matrix = np.zeros((len(SEGMENTS), len(hm_cols)))
for i, (plat, grp) in enumerate(SEGMENTS):
    seg = combined[(combined['platform'] == plat) & (combined['group'] == grp)]
    if len(seg) == 0:
        continue
    for j, key in enumerate(hm_cols):
        matrix[i, j] = seg[key].mean() * 100  # % of segment posts

hm_df = pd.DataFrame(matrix, index=seg_labels, columns=col_labels)
print('Heatmap matrix (% of posts mentioning topic):')
print(hm_df.round(2).to_string())


In [ ]:
fig, ax = styled_fig(figsize=(13, 6))

# Split colormap: blue half for Harris topics, red half for Trump topics
from matplotlib.colors import LinearSegmentedColormap

# Unified heatmap with diverging column colouring via column separator
im = ax.imshow(matrix, cmap='YlOrRd', aspect='auto', vmin=0)

ax.set_xticks(range(len(col_labels)))
ax.set_yticks(range(len(seg_labels)))
ax.set_xticklabels(col_labels, rotation=30, ha='right', fontsize=8.5, color=TEXT_PRIMARY)
ax.set_yticklabels(seg_labels, fontsize=9, color=TEXT_PRIMARY)

# Value annotations
for i in range(len(SEGMENTS)):
    for j in range(len(hm_cols)):
        v = matrix[i, j]
        ax.text(j, i, f'{v:.1f}%', ha='center', va='center', fontsize=8,
                color='black' if v > matrix.max() * 0.5 else TEXT_PRIMARY)

# Dividing line between Harris and Trump columns
ax.axvline(TOP_HM - 0.5, color=TEXT_MUTED, linewidth=2, linestyle='--', alpha=0.6)

# Column group labels
ax.text(TOP_HM / 2 - 0.5, -1.1, 'Harris agenda topics',
        ha='center', color=DEMOCRAT, fontsize=10, fontweight='bold',
        transform=ax.get_xaxis_transform())
ax.text(TOP_HM + TOP_HM / 2 - 0.5, -1.1, 'Trump agenda topics',
        ha='center', color=REPUBLICAN, fontsize=10, fontweight='bold',
        transform=ax.get_xaxis_transform())

# Dividing lines between platforms (rows 2 and 4)
ax.axhline(1.5, color=TEXT_MUTED, linewidth=1.2, alpha=0.5)
ax.axhline(3.5, color=TEXT_MUTED, linewidth=1.2, alpha=0.5)

cbar = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
cbar.set_label('% of posts mentioning topic', color=TEXT_MUTED, fontsize=9)
cbar.ax.yaxis.set_tick_params(color=TEXT_MUTED, labelcolor=TEXT_MUTED)

ax.set_title('Agenda Amplification: Which Platform Talks About Which Topics?',
             color=TEXT_PRIMARY, fontsize=13)

plt.tight_layout()
plt.savefig('../../Data/2_Silver/agenda_platform_heatmap.png',
            dpi=150, bbox_inches='tight', facecolor=BG_DARK)
plt.show()


**Insight — Platform × topic heatmap:**  
Dark cells on the Harris-topic columns in Reddit/HarrisBuzz or News/HarrisBuzz confirm **selective amplification**: 
left-leaning audiences prioritise Harris's agenda topics, right-leaning audiences prioritise Trump's.  
Cross-partisan cells (e.g. TrumpBuzz mentioning Harris's abortion topic) reveal **contested territory** — 
topics the opposing side still discusses (often critically).


## 5. Framing Divergence


Same topic, completely different vocabulary.  
We filter posts that mention **economy** or **immigration** keywords, then run TF-IDF separately  
on TrumpBuzz vs HarrisBuzz content. The most distinctive words reveal **how each side frames the issue**.


In [ ]:
# Pick topics that appear in BOTH candidates' agenda for comparison
# Economy: trump_economy_trade vs harris_economy
# Immigration: trump_immigration vs harris_safety_justice (contains immigration terms)

COMPARE_PAIRS = [
    {
        'label':       'Economy',
        'trump_key':   next((k for k in trump_topics if 'economy' in k or 'trade' in k), trump_topics[0]),
        'harris_key':  next((k for k in harris_topics if 'economy' in k), harris_topics[0]),
    },
    {
        'label':       'Immigration / Safety',
        'trump_key':   next((k for k in trump_topics if 'immigr' in k), trump_topics[0]),
        'harris_key':  next((k for k in harris_topics if 'safety' in k or 'justice' in k), harris_topics[0]),
    },
]

for p in COMPARE_PAIRS:
    print(f"{p['label']}:  Trump key = {p['trump_key']}  |  Harris key = {p['harris_key']}")


In [ ]:
def top_tfidf_words(texts, top_n=15, extra_stops=None):
    """TF-IDF distinctive words from a list of documents (treated as one corpus)."""
    stops = list(extra_stops or [])
    vec = TfidfVectorizer(
        max_features=3000, stop_words='english',
        ngram_range=(1, 1), min_df=2, sublinear_tf=True,
    )
    # Treat all texts as one big document split into chunks for IDF
    try:
        X = vec.fit_transform(texts)
    except ValueError:
        return []
    mean_scores = np.asarray(X.mean(axis=0)).flatten()
    top_idx = mean_scores.argsort()[::-1]
    words_all = vec.get_feature_names_out()
    result = []
    for i in top_idx:
        w = words_all[i]
        if w in stops or len(w) < 3:
            continue
        result.append((w, mean_scores[i]))
        if len(result) == top_n:
            break
    return result

REMOVE = ['trump', 'harris', 'kamala', 'donald', 'president', 'biden',
          'like', 'just', 'people', 'say', 'said', 'will', 'one',
          'republican', 'democrat', 'political', 'vote', 'voter']

framing_results = {}
for pair in COMPARE_PAIRS:
    lbl = pair['label']
    # Posts that mention EITHER topic key
    mask = combined[pair['trump_key']] | combined[pair['harris_key']]
    relevant = combined[mask]

    trump_texts  = relevant[relevant['group'] == 'TrumpBuzz' ]['text_clean'].dropna().tolist()
    harris_texts = relevant[relevant['group'] == 'HarrisBuzz']['text_clean'].dropna().tolist()

    framing_results[lbl] = {
        'trump':  top_tfidf_words(trump_texts,  extra_stops=REMOVE),
        'harris': top_tfidf_words(harris_texts, extra_stops=REMOVE),
        'n_trump':  len(trump_texts),
        'n_harris': len(harris_texts),
    }
    print(f'{lbl}: {len(trump_texts):,} Trump posts, {len(harris_texts):,} Harris posts')


In [ ]:
n_pairs = len(COMPARE_PAIRS)
fig, axes = plt.subplots(n_pairs, 2, figsize=(14, 5 * n_pairs), facecolor=BG_DARK)
if n_pairs == 1:
    axes = [axes]

for row, pair in enumerate(COMPARE_PAIRS):
    lbl = pair['label']
    res = framing_results[lbl]

    for col, (buzz, color, data_key) in enumerate([
        ('TrumpBuzz',  REPUBLICAN, 'trump'),
        ('HarrisBuzz', DEMOCRAT,   'harris'),
    ]):
        ax = axes[row][col]
        ax.set_facecolor(BG_PANEL)

        word_scores = res[data_key][:12]
        if not word_scores:
            ax.text(0.5, 0.5, 'Not enough data', ha='center', va='center',
                    color=TEXT_MUTED, transform=ax.transAxes)
            continue

        words  = [w for w, _ in word_scores]
        scores = [s for _, s in word_scores]
        y_pos  = range(len(words))

        bars = ax.barh(y_pos, scores, color=color, alpha=0.85)
        ax.set_yticks(y_pos)
        ax.set_yticklabels(words, fontsize=10, color=TEXT_PRIMARY)
        ax.invert_yaxis()
        ax.set_xlabel('Mean TF-IDF score', color=TEXT_MUTED, fontsize=9)
        ax.tick_params(colors=TEXT_MUTED)
        ax.grid(axis='x', color=GRID_COLOR, linewidth=0.6)
        ax.set_axisbelow(True)
        for sp in ['top', 'right']:
            ax.spines[sp].set_visible(False)
        for sp in ['left', 'bottom']:
            ax.spines[sp].set_edgecolor(SPINE_COLOR)

        n = res[f'n_{data_key}']
        ax.set_title(f'{lbl} — {buzz}  (n={n:,})',
                     color=color, fontsize=11, fontweight='bold')

fig.suptitle('Framing Divergence: Same Topic, Different Vocabulary',
             color=TEXT_PRIMARY, fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../../Data/2_Silver/agenda_framing_divergence.png',
            dpi=150, bbox_inches='tight', facecolor=BG_DARK)
plt.show()


**Insight — Framing divergence:**  
When TrumpBuzz and HarrisBuzz discuss the *same policy topic*, they use entirely different vocabulary.  
On **economy**: TrumpBuzz words likely cluster around tariffs, inflation, China, jobs; 
HarrisBuzz around middle class, opportunity, tax cuts for workers.  
On **immigration**: TrumpBuzz uses invasion, border, illegal; HarrisBuzz uses reform, asylum, families.  
This is linguistic evidence of **parallel realities** — the same issue, two incompatible narratives.


## 6. Summary


In [ ]:
print('=' * 60)
print('  AGENDA ANALYSIS — SUMMARY')
print('=' * 60)
print()
print(f'  Harris agenda sections : {len(agenda["harris"]):>4}')
print(f'  Trump  agenda sections : {len(agenda["trump"]):>4}')
print(f'  Total posts analysed   : {len(combined):>9,}')
print(f'    Reddit               : {(combined["platform"]=="Reddit").sum():>9,}')
print(f'    Bluesky              : {(combined["platform"]=="Bluesky").sum():>9,}')
print(f'    News                 : {(combined["platform"]=="News").sum():>9,}')
print()
print('  Most-mentioned Harris topic :')
best_h = hits[harris_topics].idxmax()
print(f'    {topics[best_h]["title"]} ({int(hits[best_h]):,} hits)')
print()
print('  Most-mentioned Trump topic  :')
best_t = hits[trump_topics].idxmax()
print(f'    {topics[best_t]["title"]} ({int(hits[best_t]):,} hits)')
print()
print('  Outputs saved to Data/2_Silver/:')
print('    agenda_salience_over_time.png')
print('    agenda_platform_heatmap.png')
print('    agenda_framing_divergence.png')
print('=' * 60)
